# Pokémon primary type (`type_1`) classification

This is a complete, top-to-bottom Colab workflow. It loads the shared CSV, validates and prepares the data, compares baseline and classification models with stratified cross-validation, tunes Extra Trees using training data only, and evaluates the selected model once on a held-out test set.

The target remains `type_1`. `name` is excluded as an identifier; `type_2` is retained as a predictor when present and missing values mean no secondary type. Ensure the Google Drive file is shared with anyone who has the link.

In [ ]:
%pip -q install gdown scikit-learn

import math
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay, f1_score
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FILE_ID = '12C1MY0dcLSouDrRCVkm3DJ2_yDQxe0xC'
CSV_PATH = 'pokemon_data.csv'
downloaded = gdown.download(id=FILE_ID, output=CSV_PATH, quiet=False)
if downloaded is None:
    raise RuntimeError(
        'Download failed. Confirm the Google Drive file is shared with anyone who has the link, then rerun this cell.'
    )

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
print('Dataset shape:', df.shape)
display(df.head())

## Validate and inspect the data

The target is cleaned consistently and rows without a target are removed. Entirely empty predictor columns are removed, while remaining missing values are imputed within each model pipeline to avoid leaking validation/test information.

In [ ]:
TARGET = 'type_1'
if TARGET not in df.columns:
    raise ValueError(f'Required target column {TARGET!r} is missing. Available columns: {df.columns.tolist()}')

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
df = df[df[TARGET].ne('')].copy()
if 'type_2' in df.columns:
    df['type_2'] = df['type_2'].fillna('none').astype(str).str.strip().str.lower()

print('Rows:', len(df), '| Classes:', df[TARGET].nunique())
display(df[TARGET].value_counts().rename('count').to_frame())
display(df.isna().sum().rename('missing values').to_frame())

## Create train and test sets

The test set is held out from model comparison and tuning. Stratification needs at least two examples of every class, so the checks below stop with a clear message if the dataset is too small or too imbalanced for this evaluation.

In [ ]:
X = df.drop(columns=[TARGET, 'name'], errors='ignore')
X = X.dropna(axis=1, how='all')
y = df[TARGET]

class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError(
        'At least one type_1 class has fewer than 2 examples. A stratified train/test split is not possible; collect more examples or combine/remove extremely rare classes.'
    )

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
print('Predictors:', X.columns.tolist())
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

# Keep at least one test example per class where possible.
test_rows = max(math.ceil(0.20 * len(df)), y.nunique())
if test_rows >= len(df) or len(df) - test_rows < y.nunique():
    raise ValueError('Not enough rows for a stratified train/test split with all classes represented.')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_rows, random_state=42, stratify=y
)

# Use up to five stratified folds, limited by the rarest class in the training split.
min_train_class_count = int(y_train.value_counts().min())
if min_train_class_count < 2:
    raise ValueError('The training split has fewer than 2 examples for a class; cross-validation cannot be run.')
n_splits = min(5, min_train_class_count)
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f'Train: {len(X_train)} rows; test: {len(X_test)} rows; CV folds: {n_splits}')

## Preprocessing and cross-validated model comparison

Imputation, scaling, and one-hot encoding are fitted independently inside each fold. Macro-F1 treats each type equally; accuracy is also reported but can favor common types.

In [ ]:
def make_preprocessor(scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    numeric_transform = Pipeline(numeric_steps)
    categorical_transform = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('numeric', numeric_transform, numeric_features),
        ('categorical', categorical_transform, categorical_features)
    ], remainder='drop')

models = {
    'Most frequent baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': Pipeline([
        ('prep', make_preprocessor(scale_numeric=True)),
        ('model', LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0))
    ]),
    'Random forest': Pipeline([
        ('prep', make_preprocessor()),
        ('model', RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight='balanced_subsample',
            random_state=42, n_jobs=1
        ))
    ]),
    'Extra trees': Pipeline([
        ('prep', make_preprocessor()),
        ('model', ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight='balanced',
            random_state=42, n_jobs=1
        ))
    ])
}

scoring = {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy', 'f1_macro': 'f1_macro'}
rows = []
for model_name, estimator in models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise'
    )
    rows.append({
        'model': model_name,
        'CV accuracy': scores['test_accuracy'].mean(),
        'CV balanced accuracy': scores['test_balanced_accuracy'].mean(),
        'CV macro-F1': scores['test_f1_macro'].mean(),
        'macro-F1 std': scores['test_f1_macro'].std()
    })
comparison = pd.DataFrame(rows).sort_values('CV macro-F1', ascending=False).reset_index(drop=True)
display(comparison.style.format({c: '{:.3f}' for c in comparison.select_dtypes('number').columns}))

## Tune Extra Trees

Randomized search uses only the training split and the stratified folds above. The search objective is macro-F1. This can take several minutes in Colab.

In [ ]:
search_pipeline = Pipeline([
    ('prep', make_preprocessor()),
    ('model', ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=1))
])
param_distributions = {
    'model__n_estimators': [200, 400, 600],
    'model__max_depth': [None, 15, 25, 40],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 0.7, 1.0]
}
search = RandomizedSearchCV(
    search_pipeline, param_distributions=param_distributions, n_iter=12,
    scoring='f1_macro', cv=cv, n_jobs=-1, random_state=42, verbose=1, refit=True
)
search.fit(X_train, y_train)
print('Best CV macro-F1:', round(search.best_score_, 4))
print('Best parameters:', search.best_params_)
best_model = search.best_estimator_

## Final held-out test evaluation

Evaluate the tuned model once on the test data and compare it with a majority-class baseline. Avoid using test results to continue tuning; use cross-validation or a new holdout for future experiments.

In [ ]:
test_predictions = best_model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
final_scores = pd.DataFrame([
    {
        'model': 'Majority baseline',
        'accuracy': accuracy_score(y_test, baseline_predictions),
        'balanced accuracy': balanced_accuracy_score(y_test, baseline_predictions),
        'macro-F1': f1_score(y_test, baseline_predictions, average='macro', zero_division=0)
    },
    {
        'model': 'Tuned Extra Trees',
        'accuracy': accuracy_score(y_test, test_predictions),
        'balanced accuracy': balanced_accuracy_score(y_test, test_predictions),
        'macro-F1': f1_score(y_test, test_predictions, average='macro', zero_division=0)
    }
])
display(final_scores.style.format({c: '{:.3f}' for c in ['accuracy', 'balanced accuracy', 'macro-F1']}))
print(classification_report(y_test, test_predictions, labels=sorted(y.unique()), zero_division=0))

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=sorted(y.unique()), xticks_rotation=45,
    cmap='Blues', colorbar=False, ax=ax
)
ax.set_title('Tuned Extra Trees — held-out test set')
plt.tight_layout()
plt.show()

## Inspect misclassified examples (optional)

Use this for interpretation only, not for repeated tuning against the same test split.

In [ ]:
errors = X_test.copy()
errors['actual_type_1'] = y_test
errors['predicted_type_1'] = test_predictions
display(errors.loc[errors['actual_type_1'] != errors['predicted_type_1']].head(25))